# Instruction Fine-Tuning Llama-2 for Medical Q&A — Reference Notebook

> **Reference notebook.** See [`instruction_finetuning.md`](./instruction_finetuning.md) for the
> instruction fine-tuning concepts (why / data format / process) behind this notebook, and
> [`fine_tuning.md`](../06-07-08-transfer-learning-fine-tuning/fine_tuning.md) for the underlying
> LoRA/QLoRA mechanics.

**Methods covered:**
- Formatting `(instruction, input, output)` triples into a Llama-2 chat-style prompt via a
  `formatting_func` passed directly to `SFTTrainer` (instead of pre-formatting a dataset column)
- QLoRA fine-tuning (`prepare_model_for_kbit_training` + `LoraConfig` + `SFTTrainer`)
- Merging LoRA adapters into the base model (`merge_and_unload()`)
- Wrapping a local Hugging Face pipeline as a LangChain-compatible LLM (`HuggingFacePipeline`) and
  composing it into an LCEL chain (`prompt | llm | StrOutputParser()`)

**Use this as a reference when:** you need copy-paste-ready code for instruction fine-tuning a causal
LM on `(instruction, input, output)` data and serving it through a LangChain pipeline.

**Don't use this as a reference for:** conversational memory (see the `create_agent` + checkpointer
pattern in [module 12](../12-langchain-p1/langchain_chains_memory_rag.ipynb)) — this notebook's Q&A
chain is single-turn.

In [ ]:
import torch

from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

import warnings
warnings.filterwarnings("ignore")

In [ ]:
dataset = load_dataset("nlpie/Llama2-MedTuned-Instructions")

train_data = dataset["train"].select(range(1000))
test_data = dataset["train"].select(range(1000, 1200))

# Each example is an (instruction, input, output) triple -- see instruction_finetuning.md for the format.
print(train_data[0])

In [ ]:
# formatting_func is called per-example by SFTTrainer -- it builds the full training string
# (instruction + input + target output) directly, no separate preprocessing pass needed.
def create_prompt(sample):
    return f"[INST]<<SYS>> {sample['instruction']}\n{sample['input']}[/INST]\n{sample['output']}"

print(create_prompt(train_data[0]))

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=False,
)

In [ ]:
base_repo = "NousResearch/Llama-2-7b-chat-hf"

tokenizer = AutoTokenizer.from_pretrained(base_repo)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    base_repo, quantization_config=bnb_config, device_map="auto", use_cache=False
)

In [ ]:
lora_config = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM")

# prepare_model_for_kbit_training handles freezing base weights, upcasting norms, and enabling
# gradient checkpointing in one call, consistently across quantized models.
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)

In [ ]:
training_args = TrainingArguments(
    output_dir="finetuned_model",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    optim="paged_adamw_32bit",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    save_strategy="epoch",
    logging_steps=10,
    num_train_epochs=3,
    max_steps=150,
    fp16=True,
    report_to="none",
)

sft_config = SFTConfig(packing=True, dataset_text_field="instruction")

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=test_data,
    peft_config=lora_config,
    formatting_func=create_prompt,   # builds the training string per-example, as defined above
)

trainer.train()

In [ ]:
trainer.save_model("finetuned_model_final")
merged_model = model.merge_and_unload()

In [ ]:
text_gen_pipeline = pipeline(
    "text-generation",
    model=merged_model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id,
    top_p=0.7,
    temperature=0.5,
)

llm = HuggingFacePipeline(pipeline=text_gen_pipeline)

In [ ]:
prompt = PromptTemplate.from_template(
    "[INST] <<SYS>>\nAnalyze the question and answer with the best option.\nHere is my question {context}[/INST]"
)

# LCEL replaces LLMChain here -- a local HuggingFacePipeline composes into a chain exactly like
# any hosted chat model.
qa_chain = prompt | llm | StrOutputParser()

In [ ]:
question = """###Question: All of the following provisions are included in the Primary health care according to the Alma Ata declaration except:
###Options:
A. Adequate supply of safe drinking water
B. Nutrition
C. Provision of free medicines
D. Basic sanitation"""

print(qa_chain.invoke({"context": question}))